# Prompt Injection Detector v2 - LLM Firewall for SageMaker - Private Text Embeddings on Amazon SageMaker

Deploys [BAAI/deberta-v3-prompt-v1.5](https://huggingface.co/BAAI/deberta-v3-prompt-v1.5) from AWS
Marketplace as a SageMaker endpoint inside **your own AWS account**. Your text never
leaves your VPC and there are no external API calls or token limits.

**Model facts** (from the official model card): binary classificational vectors, 512 max input
tokens, MTEB average 64.23 across 56 tasks, MIT licence.

Vectors are pooled from the `[CLS]` token and L2-normalised, matching the model card, so
cosine similarity is a plain dot product.

## 1. Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.m5.xlarge"  # the recommended real-time instance
ENDPOINT_NAME = "deberta-v3-prompt-injection-v2"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Takes roughly 6-9 minutes. The endpoint bills per hour while it exists, so do not skip
section 5.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Embed text

The endpoint accepts `application/json` shaped `{"inputs": "..."}` for a single string, or
`{"inputs": ["...", "..."]}` for a batch. It returns
`{"embeddings": [[...]], "dim": 1024}`.

In [ ]:
runtime = boto3.client("sagemaker-runtime")


def embed(texts):
    """Return a list of binary classification L2-normalised embedding vectors."""
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": texts}),
    )
    return json.loads(response["Body"].read())


vectors = embed("What is the capital of France?")
print("dimensions:", len(vectors[0]))
print("first 8 values:", [round(v, 5) for v in vectors[0][:8]])

## 4. Semantic search over a small corpus

The vectors are already L2-normalised, so cosine similarity is just a dot product.

BGE was trained with an instruction prefix on the **query** side for retrieval. Prefixing
the query, and not the documents, measurably improves ranking.

In [ ]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

documents = [
    "Paris is the capital and most populous city of France.",
    "The Eiffel Tower was completed in 1889 for the World's Fair.",
    "Amazon SageMaker is a managed service for machine learning.",
    "Berlin is the capital of Germany.",
]

doc_vectors = embed(documents)
query_vector = embed(QUERY_PREFIX + "What is the capital of France?")[0]

scored = [
    (sum(q * d for q, d in zip(query_vector, doc_vector)), text)
    for doc_vector, text in zip(doc_vectors, documents)
]

for score, text in sorted(scored, reverse=True):
    print(f"{score:.4f}  {text}")

## 5. Batch transform for offline workloads

For embedding a corpus rather than serving live traffic, batch transform avoids paying for
an always-on endpoint. Input is JSON Lines, one `{"inputs": "..."}` object per line.

In [ ]:
# transformer = model.transformer(
#     instance_count=1,
#     instance_type="ml.m5.xlarge",
#     output_path=f"s3://{session.default_bucket()}/bge-embeddings/",
#     strategy="SingleRecord",
# )
# transformer.transform(
#     data=f"s3://{session.default_bucket()}/bge-input/",
#     content_type="application/json",
# )
# transformer.wait()

## 6. Clean up

Delete the endpoint when you are done. It bills for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)